# Performance Measures for NER

*Chapter 4 - Multilingual Named Entity Recognition · Natural Language Processing with Transformers (Tunstall, von Werra and Wolf)*

## Overview

Before we train the model, we must decide **how to measure it**. For NER, the usual accuracy per token is not a good choice. Most tokens have the label `O`, so a model can get a high accuracy and still miss most of the entities. NER is judged on the level of the **entity**: an entity is correct only if the model finds the complete span of words and the correct type.

This short notebook has two parts. First, it uses the `seqeval` library on a small toy example to show how entity-level precision, recall and F1 work. Second, it defines `align_predictions`, a helper that turns the output of the model into the tag lists that `seqeval` needs.

## Objective

- Understand entity-level precision, recall and F1.
- Read a `seqeval` classification report.
- Write `align_predictions`, which removes the positions with the label `-100` and converts ids to tag names.

## Tools and inputs

- **`seqeval`** (`classification_report`, and `f1_score` in later notebooks): evaluation for sequence labeling.
- **`numpy`** for `argmax` and array indexing.
- **`index2tag`** from `tokenizing_text.ipynb`, which maps tag ids to names.

## Notes for the reader

- `import_ipynb` runs `tokenizing_text.ipynb` (and the first notebook) to get `index2tag`, so the import takes some time.
- The function `align_predictions` is used by `compute_metrics` in the fine-tuning notebooks.

In [1]:
import os
os.environ["USE_TF"] = "0"
from seqeval.metrics import classification_report

import import_ipynb
from tokenizing_text import index2tag

## Section 1 - Entity-Level Metrics with `seqeval`

### 1.1 Definitions

`seqeval` first reads the tag sequence and finds the entities. A predicted entity counts as a **true positive (TP)** only if its span and its type are both the same as a true entity. Then:

$$
\text{Precision} = \frac{TP}{TP + FP}, \qquad
\text{Recall} = \frac{TP}{TP + FN}, \qquad
F1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

- $FP$ (false positives) are predicted entities that are not in the true data.
- $FN$ (false negatives) are true entities that the model did not predict.
- The F1 score is the harmonic mean of precision and recall. It is the main number we report in the next notebooks.

### 1.2 A Toy Example

The two lists below have two sentences each. In the first sentence, the true `MISC` entity covers three tokens (positions 3 to 5, counting from 0). The model predicts a `MISC` entity that covers four tokens (positions 2 to 5), so the start is wrong. In the second sentence, the `PER` entity is predicted correctly.

In [2]:
# Toy example: the first sentence has a MISC entity, the second has a PER entity
y_true = [["O", "O", "O", "B-MISC", "I-MISC", "I-MISC", "O"],
          ["B-PER", "I-PER", "O"]]
# In the first sentence, the predicted MISC entity starts one token too early
y_pred = [["O", "O", "B-MISC", "I-MISC", "I-MISC", "I-MISC", "O"],
          ["B-PER", "I-PER", "O"]]

### 1.3 Reading the Report

The partly correct `MISC` entity does not count, because the span is not exactly the same. Only the `PER` entity is a true positive.

| Entity type | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| `MISC` | 0.00 | 0.00 | 0.00 | 1 |
| `PER` | 1.00 | 1.00 | 1.00 | 1 |
| micro avg | 0.50 | 0.50 | 0.50 | 2 |
| macro avg | 0.50 | 0.50 | 0.50 | 2 |
| weighted avg | 0.50 | 0.50 | 0.50 | 2 |

The **micro average** counts all entities together: there are 1 TP, 1 FP and 1 FN, so precision and recall are both $1/2$. The **macro average** is the simple mean of the scores of each type, and the **weighted average** uses the support (the number of true entities) of each type as the weight. In this small example, all three averages are equal.

In [3]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

        MISC       0.00      0.00      0.00         1
         PER       1.00      1.00      1.00         1

   micro avg       0.50      0.50      0.50         2
   macro avg       0.50      0.50      0.50         2
weighted avg       0.50      0.50      0.50         2



## Section 2 - Aligning Predictions with Labels

The model gives logits for **tokens**, but `seqeval` needs a list of **tags for words**. The function `align_predictions` makes this change for a whole batch:

1. Take the `argmax` over the class axis to get the predicted class of each token.
2. Go through each sentence and keep only the positions where the label id is not `-100`. These are the first sub-tokens of the words (see `tokenizing_text.ipynb`).
3. Convert the kept ids to tag names with `index2tag`.

The function returns the predicted tags first and the true tags second: `(preds_list, labels_list)`. The fine-tuning notebook uses this order.

| Input | Shape |
|---|---|
| `predictions` (logits) | `[batch_size, sequence_length, num_labels]` |
| `label_ids` | `[batch_size, sequence_length]` |

In [4]:
import numpy as np

def align_predictions(predictions, label_ids):
    """Convert model logits and label ids into lists of tag names.

    Positions whose label id is -100 (special tokens and continuation
    sub-tokens) are skipped, so each sentence has one tag for every word.

    Args:
        predictions: Array of logits with the shape
            `[batch_size, sequence_length, num_labels]`.
        label_ids: Integer array with the shape `[batch_size, sequence_length]`.
            It holds the true tag ids, and -100 for positions to ignore.

    Returns:
        A tuple `(preds_list, labels_list)`. Both are lists with one list of
        tag names for each sentence: the predicted tags and the true tags.

    Notes:
        The global dictionary `index2tag` maps tag ids to tag names.
    """
    # Choose the most likely class for every token
    preds = np.argmax(predictions, axis=2)
    batch_size, seq_len = preds.shape
    preds_list, labels_list = [], []

    for batch_idx in range(batch_size):
        example_labels, example_preds = [], []
        for seq_idx in range(seq_len):
            # Keep only the positions that have a real label
            if label_ids[batch_idx, seq_idx] != -100:
                example_labels.append(index2tag[label_ids[batch_idx][seq_idx]])
                example_preds.append(index2tag[preds[batch_idx][seq_idx]])

        preds_list.append(example_preds)
        labels_list.append(example_labels)

    return preds_list, labels_list

## Conclusion

This notebook defined the metric for the whole chapter: **entity-level F1** from `seqeval`. The toy report shows that a prediction with the right type but a wrong span gives no credit.

| Item | Result |
|---|---|
| Metric | Entity-level precision, recall and F1 (`seqeval`) |
| Toy example, `MISC` (span off by one token) | Precision 0.00, recall 0.00, F1 0.00 |
| Toy example, `PER` (exact match) | Precision 1.00, recall 1.00, F1 1.00 |
| Toy example, micro average | 0.50 for precision, recall and F1 |
| Helper | `align_predictions` returns `(preds_list, labels_list)` without `-100` positions |

### What comes next

`fine-tuning-xlm-roberta.ipynb` uses `align_predictions` inside `compute_metrics`, so the F1 score is calculated at the end of each training epoch.

## **Resources & References**

**Official Documentation**

- [`seqeval` repository (sequence labeling evaluation)](https://github.com/chakki-works/seqeval)
- [Hugging Face Transformers: Token classification guide (evaluation with `seqeval`)](https://huggingface.co/docs/transformers/tasks/token_classification)

**Additional Reading**

- Tunstall, L., von Werra, L., Wolf, T. (2022). [Natural Language Processing with Transformers, Revised Edition](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/). O'Reilly Media. Chapter 4: Multilingual Named Entity Recognition.